# 5. Base documentaire de l'entreprise (RAG)

L'IA ne connaît pas vos procédures, vos guides ni vos AMDEC passées. Ici, on lui **donne ces connaissances** :

1. on découpe vos documents en courts extraits (PDF, Word, Excel, CSV, texte) ;
2. à chaque question, on retrouve les extraits les plus pertinents ;
3. on les envoie au modèle, qui **cite ses sources** ([1], [2]…).

Le modèle n'est pas modifié : il suffit d'ajouter des documents dans le dossier pour enrichir la base.

> Les documents de `examples/documents/` sont **fictifs**. Remplacez `DOSSIER_DOCS` par votre dossier.

In [1]:
# Rend le package safety_agent importable depuis le dossier notebooks/
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
EXAMPLES = ROOT / "examples"

import pandas as pd
pd.set_option("display.max_colwidth", 120)

## 1. Construire la base

In [2]:
from safety_agent.knowledge import KnowledgeBase

DOSSIER_DOCS = EXAMPLES / "documents"     # <- votre dossier de procédures, AMDEC passées, REX...

kb = KnowledgeBase.build([DOSSIER_DOCS], on_progress=lambda f: print("lecture", f))
pd.DataFrame(kb.documents)[["path", "chunks"]]

lecture amdec_bms_gen1.xlsx
lecture guide_redaction_exigences.docx
lecture procedure_amdec.pdf
lecture retours_experience.md


,path,chunks
0,amdec_bms_gen1.xlsx,5
1,guide_redaction_exigences.docx,6
2,procedure_amdec.pdf,2
3,retours_experience.md,4


In [3]:
# Sauvegarde : la base est réutilisable sans relire les documents.
# Le fichier garde l'empreinte SHA-256 de chaque document (gestion de configuration ISO 26262).
kb.save(ROOT / "base_documentaire.json")
print(len(kb.chunks), "extraits enregistrés")

17 extraits enregistrés


## 2. Chercher dans la base (sans IA)

In [4]:
def chercher(question, k=3):
    for n, c in enumerate(kb.search(question, k), 1):
        print(f"[{n}] {c.citation()}\n    {c.text[:250]}\n")

chercher("dérive du capteur de courant en température")

[1] amdec_bms_gen1.xlsx — ligne 2
    ID: G1-01 | Élément: Capteur de courant pack | Fonction: Mesurer le courant batterie | Mode de défaillance: Offset de mesure de courant | Effet: Erreur d'estimation de l'état de charge, surcharge possible | Cause: Dérive de l'offset du capteur à effe

[2] retours_experience.md — REX-2023-07 : Dérive du capteur de courant à froid
    [REX-2023-07 : Dérive du capteur de courant à froid] Sur le projet BMS Gen1, des essais à -30 °C ont montré une dérive de l'offset du capteur de courant à effet Hall de 2 A. L'état de charge était surestimé de 4 % après 8 heures. Mesure corrective : 

[3] amdec_bms_gen1.xlsx — ligne 4
    ID: G1-03 | Élément: Capteur de courant pack | Fonction: Mesurer le courant batterie | Mode de défaillance: Perte d'alimentation du capteur | Effet: Mesure de courant à zéro, état de charge figé | Cause: Rupture du fil d'alimentation 5 V | S: 8 | O: 



In [5]:
chercher("temps de réaction à une perte de communication CAN")

[1] guide_redaction_exigences.docx — 2. Exigences temporelles
    [2. Exigences temporelles] R4. Toute réaction à un défaut est exprimée en millisecondes et doit être compatible avec le FTTI de l'objectif de sécurité parent.
R5. Pour le BMS, le temps de réaction maximal à une perte de communication CAN est de 50 ms

[2] guide_redaction_exigences.docx — 5. Exemples
    [5. Exemples] À éviter | Formulation conforme
Le BMS doit réagir rapidement en cas de défaut CAN. | Le BMS doit détecter une perte de communication CAN en moins de 50 ms.
Le BMS doit ouvrir les contacteurs et doit enregistrer un défaut. | FSR-a : Le 

[3] amdec_bms_gen1.xlsx — ligne 5
    ID: G1-04 | Élément: Communication CAN | Fonction: Transmettre l'état batterie au VCU | Mode de défaillance: Trame figée (compteur de vie bloqué) | Effet: Le VCU utilise des valeurs anciennes de limites de puissance | Cause: Blocage de la tâche logic



## 3. Utiliser la base avec l'IA

In [6]:
# Configuration du modèle hébergé (Ollama par défaut)
BASE_URL = "http://localhost:11434/v1"   # vLLM : "http://serveur:8000/v1"
MODEL = "qwen2.5:7b"     # qwen2.5:14b ou plus si vous avez une carte graphique

from safety_agent.notebook import get_llm
llm = get_llm(BASE_URL, MODEL)   # bascule en MODE DÉMO si le serveur ne répond pas

⚠️  Serveur LLM injoignable à http://localhost:11434/v1 (<urlopen error [Errno 111] Connection refused>)
➡️  MODE DÉMO : réponses simulées, sans IA. Lancez Ollama/vLLM pour de vraies analyses.


In [7]:
from safety_agent.loaders import load_project

# Remplacez par vos fichiers (.csv ou .xlsx)
REQS = EXAMPLES / "bms_exigences.csv"
FMEA = EXAMPLES / "bms_amdec.csv"

project, load_findings = load_project(REQS, FMEA)
print(f"{len(project.requirements)} exigences, {len(project.fmea)} lignes AMDEC")

11 exigences, 6 lignes AMDEC


### Relecture d'exigence appuyée sur votre guide de rédaction

In [8]:
from safety_agent.ai import review_requirement
from safety_agent.notebook import findings_df, proposals_df

findings, proposals = review_requirement(llm, project, project.requirements["FSR-06"], kb=kb)
display(findings_df(findings))
for p in proposals:
    print("Proposition :", p.content)
    print("Justification :", p.rationale)
    print("Sources envoyées au modèle :", *p.data["sources"], sep="\n  ")

,Code,Niveau,Élément,Constat,Source
0,IA-AMBIG,warning,FSR-06,"Exigence ambiguë : termes non définis : appropriée, rapidement",IA
1,IA-ATOM,warning,FSR-06,Exigence non atomique : plusieurs obligations dans une même exigence,IA
2,IA-VERIF,warning,FSR-06,Exigence non vérifiable : pas de critère de réussite chiffré,IA


Proposition : [DÉMO] Reformuler avec une valeur chiffrée (ex. temps de réaction en ms) et un état sûr défini.
Justification : Réponse simulée (mode démo), construite à partir des règles automatiques. Voir aussi la base documentaire : [1] guide_redaction_exigences.docx — 5. Exemples.
Sources envoyées au modèle :
  [1] guide_redaction_exigences.docx — 5. Exemples
  [2] guide_redaction_exigences.docx — 3. États sûrs
  [3] guide_redaction_exigences.docx — 2. Exigences temporelles
  [4] retours_experience.md — REX-2023-07 : Dérive du capteur de courant à froid


### Modes de défaillance enrichis par vos AMDEC passées

In [9]:
from safety_agent.ai import suggest_failure_modes

props = suggest_failure_modes(llm, project, "Capteur de courant pack", "Mesurer le courant batterie",
                              "BMS 400 V, véhicule électrique", kb=kb)
pd.DataFrame([{"Mode": p.data.get("mode"), "S": p.data.get("S"), "O": p.data.get("O"), "D": p.data.get("D"),
               "Justification": p.rationale} for p in props])

,Mode,S,O,D,Justification
0,[DÉMO] Perte du signal (Capteur de courant pack),8,3,3,"Exemple générique (mode démo), à remplacer par l'analyse du modèle."
1,[DÉMO] Valeur figée (Capteur de courant pack),9,3,5,"Exemple générique (mode démo), à remplacer par l'analyse du modèle."
2,[DÉMO] Valeur erronée hors tolérance (Capteur de courant pack),9,4,5,"Exemple générique (mode démo), à remplacer par l'analyse du modèle."
3,[DÉMO] Perte d'alimentation du capteur,8,3,3,Repris de l'AMDEC d'un projet précédent [1] amdec_bms_gen1.xlsx — ligne 4.
4,[DÉMO] Offset de mesure de courant,9,4,4,Repris de l'AMDEC d'un projet précédent [2] amdec_bms_gen1.xlsx — ligne 2.
5,[DÉMO] Saturation de la mesure,10,2,5,Repris de l'AMDEC d'un projet précédent [3] amdec_bms_gen1.xlsx — ligne 3.


### L'agent consulte la base quand c'est utile

In [10]:
from safety_agent.agent import SafetyAgent

agent = SafetyAgent(llm, project, kb=kb, on_tool=lambda n, a: print(f"  🔧 {n} {a or ''}"))
print(agent.ask("Que disent nos retours d'expérience sur le test du watchdog au démarrage ?"))

  🔧 search_knowledge {'query': "Que disent nos retours d'expérience sur le test du watchdog au démarrage ?"}
[DÉMO — réponse simulée, pas d'IA] Extraits trouvés dans la base documentaire :
- retours_experience.md — REX-2024-02 : Réveil intempestif par le watchdog : [REX-2024-02 : Réveil intempestif par le watchdog] Le watchdog externe n'était pas testé au démarrage. Un défaut de configuration est passé inaperçu jusqu'aux essais véhicule. Mesure corrective : test de déclenchement du…
- retours_experience.md — Retours d'expérience sûreté de fonctionnement — BMS : [Retours d'expérience sûreté de fonctionnement — BMS] Document d'exemple fictif, créé pour la démonstration de safety-agent.…
- procedure_amdec.pdf — page 1 : Revue L'AMDEC est revue en groupe pluridisciplinaire (conception, sûreté, test, qualité) à chaque jalon du projet.…
- amdec_bms_gen1.xlsx — ligne 6 : ID: G1-05 | Élément: Contacteur principal | Fonction: Isoler la batterie | Mode de défaillance: Contacteur collé fermé | Ef

## 4. Option : recherche sémantique (meilleure qualité)

Par défaut, la recherche se fait **par mots-clés** : rapide, sans modèle, suffisante pour démarrer.
Pour retrouver aussi les passages formulés autrement (synonymes, paraphrases), ajoutez un **modèle d'embeddings** :

```
ollama pull bge-m3
```

(≈ 1,2 Go, multilingue, bon en français), puis reconstruisez la base :

```python
kb = KnowledgeBase.build([DOSSIER_DOCS], llm=llm, embed_model="bge-m3")
```

Les deux recherches sont alors combinées automatiquement.